#### Parameter-efficient Finetuning with LoRA

Low-rank adaptation (LoRA) is a machine learning technique that modifies a pretrained model to better suit a specific, often smaller, dataset by adjusting only a small, 

low-rank subset of the model's parameters

**NOTE1**: why we initialize elememnts of A matrix with kaiming_uniform

The Kaiming initialization method is calculated as a random number with a Gaussian probability distribution (G) with a mean of 0.0 and a standard deviation of sqrt(2/n), 

where n is the number of inputs to the node.

**NOTE2**: nn.Parameter biến một tensor thường thành tham số có thể học được (learnable parameter) của mô hình

- Bước 1: khởi tạo 1 ma trận A rỗng có kích thước (in_dim, rank): khi nào cần khởi tạo trọng số theo phân phối sử dụng torch.empty(), ngược lại thì torch.zeros()

    rank là số chiều của không gian con (subspace dimension) dùng để xấp xỉ ma trận lớn . Rank càng nhỏ thì số tham số càng ít, giúp tiết kiệm bộ nhớ và tăng tốc huấn 
    
    luyện. Ví dụ: thay vì học ma trận  (1024, 1024) , bạn học hai ma trận nhỏ hơn  A: (1024, 8)  và  B: (8, 1024)  với rank = 8.

- Bước 2: khởi tạo trọng số ban đầu cho ma trận bằng phân phối kaiming: Tránh vanishing/exploding gradients + Tăng tốc hội tụ + Cải thiện độ chính xác

    Tham số  a  đại diện cho hệ số góc âm của LeakyReLU

In [ ]:
import math
import torch
import torch.nn.init as init

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank))
        torch.nn.init.kaiming_uniform_(self.A, a = math.sqrt(5)) # khởi tạo trọng số cho ma trận A
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha
        self.rank = rank
    def forward(self, X):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

In [ ]:

class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [ ]:

def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            # Replace the Linear layer with LinearWithLoRA
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            # Recursively apply the same function to child modules
            replace_linear_with_lora(module, rank, alpha)

In [ ]:

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

for param in model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after: {total_params:,}")

In [ ]:
replace_linear_with_lora(model, rank=16, alpha=16)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")

In [ ]:
model.to(device)
print(model)